In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

# from langchain_openai import ChatOpenAI

# # Initialize the model
# model = ChatOpenAI(
#     model="gpt-4o", 
#     temperature=0.7
# )

True

In [3]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=1.0,
    max_retries=2,
)

In [4]:
class BlogState(TypedDict):
    
    title: str
    outline: str
    content: str

In [5]:
def create_outline(state: BlogState) -> BlogState:
    
    # Fetch title
    title = state['title']
    
    # Call llm gen outline
    prompt = f'Generate the detailed outline for a blog on the topic: "{title}".'
    outline = model.invoke(prompt)
    
    # Update State
    state['outline'] = outline
    
    return state

In [6]:
def create_blog(state: BlogState) -> BlogState:
    
    title = state['title']
    outline = state['outline']
    
    prompt = f'Write a detailed blog on the title - {title} using the following outline: {outline}.'
    
    blog = model.invoke(prompt).content
    
    state['content'] = blog
    
    return state

In [7]:
graph = StateGraph(BlogState)

# Nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# Edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)


workflow = graph.compile()

In [8]:
initial_state = { 'title': 'Fundamental and advance topics in python!'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Fundamental and advance topics in python!', 'outline': AIMessage(content=[{'type': 'text', 'text': 'This outline is designed to provide a comprehensive roadmap for a blog post (or a series of posts) that caters to both beginners starting their journey and experienced developers looking to sharpen their skills.\n\n---\n\n# Blog Title Idea: From Script to Scale: Mastering Python from Fundamental to Advanced\n\n## Introduction\n*   **The Python Phenomenon:** Why Python remains the #1 language (versatility, community, and ease of use).\n*   **Target Audience:** Bridging the gap between "Hello World" and "Production-Ready Code."\n*   **The Roadmap:** A brief overview of what the reader will achieve by the end of the post.\n\n---\n\n## Part 1: The Foundations (The "Must-Haves")\n*Goal: Ensuring a rock-solid understanding of the basics.*\n\n*   **1.1 Data Structures Under the Hood:**\n    *   Lists vs. Tuples vs. Sets vs. Dictionaries (When and why to use each).\n    *   Understand

In [9]:
print(final_state['outline'])

content=[{'type': 'text', 'text': 'This outline is designed to provide a comprehensive roadmap for a blog post (or a series of posts) that caters to both beginners starting their journey and experienced developers looking to sharpen their skills.\n\n---\n\n# Blog Title Idea: From Script to Scale: Mastering Python from Fundamental to Advanced\n\n## Introduction\n*   **The Python Phenomenon:** Why Python remains the #1 language (versatility, community, and ease of use).\n*   **Target Audience:** Bridging the gap between "Hello World" and "Production-Ready Code."\n*   **The Roadmap:** A brief overview of what the reader will achieve by the end of the post.\n\n---\n\n## Part 1: The Foundations (The "Must-Haves")\n*Goal: Ensuring a rock-solid understanding of the basics.*\n\n*   **1.1 Data Structures Under the Hood:**\n    *   Lists vs. Tuples vs. Sets vs. Dictionaries (When and why to use each).\n    *   Understanding Mutability and Memory implications.\n*   **1.2 Control Flow & Logic:**\n